# Stage 09 — narrow public-data pretraining pilot

## tl;dr

**Prepared; GPU results are not yet available.** This notebook compares an official-only control with two epochs of admitted external authorship pretraining followed by the identical official training schedule. Both start from the same untouched pinned DeBERTa-v3-base vendor checkpoint; the single classification head is reset identically before both official phases.

The official S03 result of 62.0% public balanced accuracy is a separate result. This pilot uses a smaller encoder, 750 external examples from five GPT labels and one reused development fold. It is not the upstream 83k training reproduction and cannot establish an ensemble gain.

Keep this notebook and its outputs **private**. Only the `aggregate/` folder is suitable for later public review after inspection; `private/` contains original-row predictions and fitted checkpoints.


## Context & Methods

The question is whether the staged procedure improves fixed-fold development balanced accuracy. Extra pretraining compute is part of the procedure, so this is not a compute-matched estimate of the value of external examples alone.

### Key Assumptions

- The corpus is frozen V4 `new`/direct-OpenAI only: 750 responses in 150 complete five-label prompt blocks. Every external epoch uses each row once, shuffled; each label has exactly 150 examples.
- Official fold 1 has 2,159 training rows and 540 validation rows, all 18 labels; whole original prompts remain disjoint. Admission is conservative, but original prompt wording is unavailable, so semantic independence is not certified.
- Architecture: attention-mask mean of the untouched encoder's last hidden state plus a single linear 18-label head. Seed 20261024, sequence length 384, batch 4, accumulation 4.
- External phase: 2 epochs, encoder LR 1.5e-5/head LR 1e-3. Official phases: 3 epochs, encoder LR 1e-5/head LR 5e-4; fresh identical head and official optimizer/scheduler. Final epoch is fixed, with no validation-based selection.
- Two allocated T4 GPUs; training uses CUDA device 0. Both allocated devices count toward a cumulative three GPU-hour cap, including setup. No paid service is used.

### 1. Verify pinned offline dependencies

Attach the official competition, prvsiyan's Extended Name That LLM Dataset **V4**, and our private Stage04B Resources **V1 359317729**. Use Python 3.13 and keep Internet off. No model or package is downloaded by this notebook.


In [ ]:
RESOURCE_MANIFEST_SHA = 'f8e60471afe49ec5f883bf141fc100d84081b2fe6a0eb28b1f9236486b7532bf'
# The budget clock starts before dependency setup, not after model loading.
from pathlib import Path
import hashlib, json, os, subprocess, sys, time
stage09_allocation_started = time.perf_counter()
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", USE_TF="0", USE_FLAX="0",
                  TOKENIZERS_PARALLELISM="false", CUBLAS_WORKSPACE_CONFIG=":4096:8")
def unique_path(pattern):
    matches = list(Path("/kaggle/input").rglob(pattern))
    if len(matches) != 1:
        raise RuntimeError(f"Expected one frozen attachment asset {pattern}; found {len(matches)}")
    return matches[0]
resource_manifest_path = unique_path("resources_manifest.json")
if hashlib.sha256(resource_manifest_path.read_bytes()).hexdigest() != RESOURCE_MANIFEST_SHA:
    raise RuntimeError("Attach the exact private Stage04B Resources V1 (359317729)")
resource_dir = resource_manifest_path.parent
resource_manifest = json.loads(resource_manifest_path.read_text())
if sys.version_info[:2] != (3, 13):
    raise RuntimeError("Pinned offline sentencepiece wheel requires Python 3.13; do not silently substitute")
wheel_paths = []
for name, expected in resource_manifest["wheel_files_sha256"].items():
    paths = list(resource_dir.rglob(name))
    if len(paths) != 1 or hashlib.sha256(paths[0].read_bytes()).hexdigest() != expected:
        raise RuntimeError("Pinned offline dependency asset mismatch: " + name)
    wheel_paths.append(str(paths[0]))
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", *wheel_paths], check=True)
train_path = unique_path("train.csv")
source_dir = unique_path("passages_clean.csv").parent
print("Pinned offline dependencies and unique required input paths resolved; no test file opened.")


## Data

### 2. Materialize the frozen audit and runner

The embedded source is deliberately explicit and portable. The CPU audit verifies immutable input hashes, public prompt/call associations, the exact 750-row admission, whole-prompt overlap screening and frozen official membership before fitting. The vendor resource manifest and every model asset are verified again.

The checks below validate accumulation arithmetic, class exposure, prediction alignment and known bootstrap contrasts on synthetic data. They do not execute Torch, AMP or a model.


In [ ]:
CODE_SOURCES = {'stage09_data_audit.py': '"""Stage 09: fixed, CPU-only admission audit for the existing direct-OpenAI V4 pilot.\n\nThis helper never trains, calls an API, serializes original rows, or certifies\nsemantic prompt independence. Its returned data frames are private in-memory\ninputs for the separate training runner. Public artifacts contain aggregates,\nexternal-data identifiers and hashes only.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport re\nimport tempfile\nimport unicodedata\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.model_selection import GroupKFold\nfrom sklearn.neighbors import NearestNeighbors\n\nTRAIN_SHA = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nSOURCE_HASHES = {\n    "passages_clean.csv": "23f90b6ba4cc83137ce8d48b3b3787439b44867532d63e2fe52f8a5ea49cd1e6",\n    "prompts.csv": "7cc4651b362ceadff4a1fd88dd1f82bad0c84d10b621a236be72c27f1b8f3a52",\n    "provenance.jsonl": "ddc69f2c221d921f928bf7785b5971d9763eaa136815c5968a83760994e405f3",\n}\nVALIDATION_IDS_SHA = "4a39f1ed94d312fc815fe69ee0d9a467fe8578628886433a92a4f72256bf4a0a"\nKNOWN_PROMPT_SOURCE = "own (written for this dataset, host-style protocol)"\nMODELS = {\n    "gpt-5.5": "gpt-5.5-2026-04-23",\n    "gpt-5.6-luna": "gpt-5.6-luna",\n    "gpt-5.6-sol": "gpt-5.6-sol",\n    "gpt-5.6-terra": "gpt-5.6-terra",\n    "gpt-6-astra": "gpt-6-astra",\n}\nDERIVATIVE = re.compile(\n    r"train[_\\s-]*regen|reconstruct(?:ed|ion)?|official[_\\s-]*prompt|"\n    r"competition[_\\s-]*prompt|derived[_\\s-]*from[_\\s-]*(?:official|train|competition)",\n    re.I,\n)\n\n\nclass AuditError(ValueError):\n    """An input, source association, or frozen pilot contract failed."""\n\n\ndef require(condition, message):\n    if not bool(condition):\n        raise AuditError(message)\n\n\ndef sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef text_sha(value):\n    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()\n\n\ndef check_hash(path, expected):\n    require(Path(path).is_file(), "Required frozen input is unavailable: " + Path(path).name)\n    require(sha(path) == expected, "Frozen input checksum mismatch: " + Path(path).name)\n\n\ndef normalize(text):\n    return " ".join(unicodedata.normalize("NFKC", str(text)).casefold().split())\n\n\ndef punctuation_normalize(text):\n    return " ".join(re.findall(r"\\w+", normalize(text)))\n\n\ndef grams(text):\n    words = punctuation_normalize(text).split()\n    return {" ".join(words[i:i + 8]) for i in range(len(words) - 7)}\n\n\ndef exclusion_blocks(prompt_rows):\n    """Conservative metadata admission; one bad record excludes its whole block."""\n    reasons = {}\n    for row in prompt_rows.itertuples():\n        bad = []\n        source = str(row.prompt_source).strip()\n        licence = str(row.prompt_licence).strip()\n        if source != KNOWN_PROMPT_SOURCE:\n            bad.append("unknown_or_unadmitted_prompt_origin")\n        if licence != "CC BY 4.0":\n            bad.append("unadmitted_prompt_licence")\n        if str(row.split) != "new":\n            bad.append("not_new_prompt_split")\n        origin_fields = " ".join(str(getattr(row, k, "")) for k in\n                                 ["prompt_source", "source_prompt_id", "prompt_id"])\n        if DERIVATIVE.search(origin_fields):\n            bad.append("explicit_official_reconstruction_or_derivative")\n        if not str(row.prompt_body).strip():\n            bad.append("empty_prompt_body")\n        if bad:\n            reasons.setdefault(str(row.prompt_id), set()).update(bad)\n    return {key: sorted(value) for key, value in sorted(reasons.items())}\n\n\ndef verify_envelope(row, envelope):\n    """Match the selected public CSV cell to its published first-party call envelope."""\n    body = envelope["request"]["body"]\n    response = envelope["response"]["json"]\n    require(envelope["response"]["http_status"] == 200, "Selected call HTTP status failed")\n    require("choices" in response and len(response["choices"]) > 0, "Wrong selected response schema")\n    answer = response["choices"][0]["message"]["content"]\n    request_prompt = body["messages"][0]["content"]\n    checks = {\n        "response": " ".join(answer.split()) == " ".join(row.text.split()),\n        "requested_model": body["model"] == row.requested_model_id,\n        "served_model": response["model"] == row.served_model_id,\n        "actual_prompt": request_prompt == row.prompt_sent,\n        "envelope_prompt": envelope["prompt_sent"] == row.prompt_sent,\n        "envelope_prompt_id": str(envelope["prompt_id"]) == str(row.prompt_id),\n        "label": envelope["label"] == row.model,\n        "route": envelope["route"] == "openai",\n        "split": envelope["split"] == "new",\n        "call_id": envelope["call_id"] == row.call_id,\n    }\n    require(all(checks.values()), "Published call evidence mismatch: " + ",".join(k for k, v in checks.items() if not v))\n    return answer == row.text\n\n\ndef frozen_split(train):\n    """Return the predeclared fold 1, checked against actual saved Stage 07A membership."""\n    require(len(train) == 2699 and train.prompt_id.nunique() == 150 and train.model.nunique() == 18,\n            "Official training dimensions changed")\n    require(train.id.is_unique, "Official row identity is not unique")\n    tr, va = next(GroupKFold(5).split(train.text, train.model, train.prompt_id))\n    val_sha = text_sha("\\n".join(train.id.iloc[va]) + "\\n")\n    require(val_sha == VALIDATION_IDS_SHA, "Fold 1 no longer matches the frozen Stage 07A membership")\n    require(set(train.prompt_id.iloc[tr]).isdisjoint(train.prompt_id.iloc[va]), "Official prompt leakage")\n    labels = sorted(train.model.unique())\n    require(set(train.model.iloc[tr]) == set(labels) == set(train.model.iloc[va]), "A pilot split lacks official labels")\n    receipt = {\n        "fold": 1,\n        "algorithm": "sklearn.model_selection.GroupKFold(n_splits=5, shuffle=False)",\n        "official_train_sha256": TRAIN_SHA,\n        "validation_id_order_sha256": val_sha,\n        "training_id_order_sha256": text_sha("\\n".join(train.id.iloc[tr]) + "\\n"),\n        "training_rows": len(tr), "validation_rows": len(va),\n        "training_prompts": int(train.prompt_id.iloc[tr].nunique()),\n        "validation_prompts": int(train.prompt_id.iloc[va].nunique()),\n        "labels": labels,\n        "training_class_support": train.model.iloc[tr].value_counts().sort_index().to_dict(),\n        "validation_class_support": train.model.iloc[va].value_counts().sort_index().to_dict(),\n        "group_disjoint": True,\n        "frozen_stage07a_validation_membership_matches": True,\n        "evaluation_role": "reused development fold; no untouched-test claim",\n        "original_ids_or_prompt_ids_published": False,\n    }\n    return tr, va, receipt\n\n\ndef scan_overlap(train, extra):\n    overlap = {}\n    normalized_official = [normalize(x) for x in train.text]\n    normalized_extra = [normalize(x) for x in extra.text]\n    suspect = np.zeros(len(extra), dtype=bool)\n    for name, fn in [("exact", lambda x: x), ("unicode_case_whitespace", normalize),\n                     ("punctuation", punctuation_normalize)]:\n        mask = extra.text.map(fn).isin(set(train.text.map(fn))).to_numpy()\n        overlap[name] = int(mask.sum())\n        suspect |= mask\n    official_grams = set().union(*(grams(t) for t in train.text))\n    shared = np.array([len(grams(t) & official_grams) for t in extra.text])\n    substring = np.array([any(a in b or b in a for a in normalized_official) for b in normalized_extra])\n    vectorizer = TfidfVectorizer(analyzer="char", ngram_range=(3, 5), min_df=2,\n                               max_features=80000, sublinear_tf=True, dtype=np.float32)\n    official_x = vectorizer.fit_transform(train.text)\n    external_x = vectorizer.transform(extra.text)\n    distance, _ = NearestNeighbors(n_neighbors=1, metric="cosine", algorithm="brute", n_jobs=2).fit(\n        official_x).kneighbors(external_x)\n    cosine = 1 - distance[:, 0]\n    suspect |= substring | (shared >= 3) | (cosine >= .85)\n    reasons = {str(prompt): ["response_overlap_or_similarity_threshold"] for prompt in\n               sorted(extra.loc[suspect, "prompt_id"].unique())}\n    result = {\n        "comparison_scope": "all 2699 official responses, labels ignored, before either arm trains",\n        "full_response_overlap_rows": overlap,\n        "substring_overlap_rows": int(substring.sum()),\n        "rows_with_at_least_three_distinct_shared_word_8grams": int((shared >= 3).sum()),\n        "rows_with_character_cosine_at_least_0_85": int((cosine >= .85).sum()),\n        "maximum_character_cosine": float(cosine.max()),\n        "maximum_shared_word_8grams": int(shared.max()),\n        "flagged_rows": int(suspect.sum()),\n        "excluded_external_prompt_blocks": sorted(reasons),\n        "screen_features_used_for_training": False,\n        "semantic_prompt_independence_certified": False,\n    }\n    return reasons, result, shared, cosine, substring\n\n\ndef self_tests(train=None):\n    """Small contract fixtures test rejection paths without fitting a classifier."""\n    passed = []\n\n    def rejected(name, fn):\n        try:\n            fn()\n        except (AuditError, KeyError):\n            passed.append(name)\n        else:\n            raise AssertionError("Failure fixture was accepted: " + name)\n\n    with tempfile.TemporaryDirectory() as td:\n        path = Path(td) / "frozen.csv"\n        path.write_text("unchanged", encoding="utf-8")\n        expected = sha(path)\n        check_hash(path, expected)\n        path.write_text("tampered", encoding="utf-8")\n        rejected("tampered_immutable_input_rejected", lambda: check_hash(path, expected))\n        rejected("missing_immutable_input_rejected", lambda: check_hash(Path(td) / "missing", expected))\n    fixture = pd.DataFrame([{"prompt_id": "external_fixture", "prompt_source": KNOWN_PROMPT_SOURCE,\n                             "prompt_licence": "CC BY 4.0", "split": "new", "prompt_body": "Fixture topic"}])\n    require(not exclusion_blocks(fixture), "Known-origin fixture failed")\n    passed.append("known_origin_admitted")\n    for name, column, value in [\n        ("unknown_origin_whole_block_excluded", "prompt_source", ""),\n        ("reconstructed_origin_whole_block_excluded", "prompt_source", "official prompt reconstruction"),\n        ("train_regen_id_whole_block_excluded", "prompt_id", "train_regen_fixture"),\n        ("unknown_licence_whole_block_excluded", "prompt_licence", "unknown"),\n        ("non_new_prompt_whole_block_excluded", "split", "train_regen"),\n        ("empty_prompt_whole_block_excluded", "prompt_body", ""),\n    ]:\n        altered = fixture.copy()\n        altered.loc[0, column] = value\n        require(len(exclusion_blocks(altered)) == 1, "Origin exclusion fixture failed: " + name)\n        passed.append(name)\n    row = pd.Series({"text": "Test answer", "requested_model_id": "test-model", "served_model_id": "test-model",\n                     "prompt_sent": "Test prompt", "prompt_id": "external_fixture", "model": "test-label", "call_id": "test-call"})\n    envelope = {"request": {"body": {"model": "test-model", "messages": [{"content": "Test prompt"}]}},\n                "response": {"http_status": 200, "json": {"model": "test-model", "choices": [{"message": {"content": "Test answer"}}]}},\n                "prompt_sent": "Test prompt", "prompt_id": "external_fixture", "label": "test-label", "route": "openai", "split": "new", "call_id": "test-call"}\n    verify_envelope(row, envelope)\n    passed.append("matching_call_envelope_admitted")\n    changed = json.loads(json.dumps(envelope))\n    changed["response"]["json"]["model"] = "wrong-model"\n    rejected("wrong_served_identity_rejected", lambda: verify_envelope(row, changed))\n    changed = json.loads(json.dumps(envelope))\n    changed["request"]["body"]["messages"][0]["content"] = "Different prompt"\n    rejected("wrong_actual_prompt_rejected", lambda: verify_envelope(row, changed))\n    if train is not None:\n        altered = train.copy()\n        _, validation, _ = frozen_split(train)\n        altered.loc[validation[0], "id"] = "stage09_split_tamper_fixture"\n        rejected("changed_actual_validation_identity_rejected", lambda: frozen_split(altered))\n    return {"status": "PASS", "checks": len(passed), "passed": passed,\n            "classifier_fitted": False, "provider_api_called": False}\n\n\ndef audit(train_path, source, out, reference_oof=None, persist=True):\n    """Return (private official frame, private admitted external frame, aggregate receipt)."""\n    train_path, source, out = Path(train_path), Path(source), Path(out)\n    check_hash(train_path, TRAIN_SHA)\n    for name, expected in SOURCE_HASHES.items():\n        check_hash(source / name, expected)\n    train = pd.read_csv(train_path, keep_default_na=False, dtype=str)\n    require(not train[["id", "prompt_id", "model", "text"]].eq("").any().any(), "Empty official input field")\n    tr, va, split = frozen_split(train)\n    reference_checked = False\n    if reference_oof is not None:\n        old = pd.read_csv(reference_oof, keep_default_na=False, dtype={"id": str, "prompt_id": str, "model": str})\n        require(train[["id", "prompt_id", "model"]].equals(old[["id", "prompt_id", "model"]]), "Saved Stage 07A identity/order mismatch")\n        for fold, (_, ix) in enumerate(GroupKFold(5).split(train.text, train.model, train.prompt_id), 1):\n            require(old.iloc[ix].fold.eq(fold).all(), "Saved Stage 07A folds differ")\n        reference_checked = True\n    table = pd.read_csv(source / "passages_clean.csv", keep_default_na=False)\n    bank = pd.read_csv(source / "prompts.csv", keep_default_na=False)\n    selected = table[table.split.eq("new") & table.provider_route.eq("openai")].copy()\n    require(len(selected) == 750 and selected.prompt_id.nunique() == 150, "Frozen external selection dimensions differ")\n    require(selected.id.is_unique and selected.call_id.is_unique, "External cell/call identity duplicate")\n    require(set(selected.model) == set(MODELS), "Selected external labels differ")\n    require(selected.groupby("model").size().eq(150).all(), "Selected external class support differs")\n    require(selected.groupby("prompt_id").model.nunique().eq(5).all(), "Incomplete five-label prompt block")\n    require(not selected.duplicated(["prompt_id", "model"]).any(), "Repeated prompt/label cell")\n    require(not selected.text.map(normalize).duplicated().any(), "Duplicated normalized external response")\n    require(not train.text.map(normalize).duplicated().any(), "Duplicated normalized official response")\n    require(selected.text.str.split().str.len().between(40, 250).all(), "External word bounds failed")\n    require(selected.finish_reason_norm.eq("stop").all(), "External completion was not a clean stop")\n    for flag in ["is_refusal", "is_truncated", "is_empty"]:\n        require(not selected[flag].any(), "Selected external quality flag: " + flag)\n    require(selected.passes_funnel.all() and selected.in_word_range.all(), "Selected external quality funnel failed")\n    for model, identifier in MODELS.items():\n        rows = selected[selected.model.eq(model)]\n        require(rows.requested_model_id.eq(identifier).all() and rows.served_model_id.eq(identifier).all(), "Requested/served identity changed")\n    require(bank.prompt_id.is_unique, "Duplicate prompt bank association")\n    missing = set(selected.prompt_id) - set(bank.prompt_id)\n    require(not missing, "Missing external prompt bank association")\n    prompt_rows = selected[["prompt_id", "source_prompt_id"]].drop_duplicates().merge(\n        bank[["prompt_id", "split", "prompt_source", "prompt_licence", "prompt_body"]],\n        on="prompt_id", validate="many_to_one")\n    origin_reasons = exclusion_blocks(prompt_rows)\n    joined = selected.merge(bank[["prompt_id", "prompt_body", "prompt_source", "prompt_licence"]],\n                            on="prompt_id", validate="many_to_one")\n    body_match = np.array([bool(body.strip()) and body in sent for body, sent in zip(joined.prompt_body, joined.prompt_sent)])\n    for prompt in joined.loc[~body_match, "prompt_id"].unique():\n        origin_reasons.setdefault(str(prompt), []).append("actual_request_prompt_not_associated_with_prompt_bank_body")\n    evidence = {}\n    calls = set(selected.call_id)\n    with (source / "provenance.jsonl").open(encoding="utf-8") as handle:\n        for line in handle:\n            envelope = json.loads(line)\n            if envelope.get("call_id") in calls:\n                require(envelope["call_id"] not in evidence, "Duplicated selected raw call evidence")\n                evidence[envelope["call_id"]] = envelope\n    require(set(evidence) == calls, "Selected raw call evidence is incomplete")\n    exact_raw = 0\n    for row in selected.itertuples():\n        exact_raw += int(verify_envelope(row, evidence[row.call_id]))\n    overlap_reasons, overlap, shared, cosine, substring = scan_overlap(train, joined)\n    excluded = set(origin_reasons) | set(overlap_reasons)\n    joined["excluded_prompt_block"] = joined.prompt_id.isin(excluded)\n    manifest = joined[["id", "call_id", "prompt_id", "model", "requested_model_id", "served_model_id",\n                       "prompt_source", "prompt_licence", "source_prompt_id", "excluded_prompt_block"]].copy()\n    manifest["response_sha256"] = joined.text.map(text_sha)\n    manifest["actual_request_prompt_sha256"] = joined.prompt_sent.map(text_sha)\n    manifest["prompt_body_sha256"] = joined.prompt_body.map(text_sha)\n    manifest["shared_official_word_8grams"] = shared\n    manifest["nearest_official_character_cosine"] = cosine\n    manifest["substring_overlap"] = substring\n    manifest = manifest.sort_values(["prompt_id", "model", "id"]).reset_index(drop=True)\n    manifest_bytes = manifest.to_csv(index=False, lineterminator="\\n", float_format="%.9g").encode("utf-8")\n    admitted = joined[~joined.excluded_prompt_block].sort_values(["prompt_id", "model"]).reset_index(drop=True)\n    require(len(admitted) > 0, "No external prompt blocks were admitted")\n    admitted_ids_sha = text_sha("\\n".join(admitted.id) + "\\n")\n    fingerprint_inputs = {\n        "official_train_sha256": TRAIN_SHA,\n        "source_files_sha256": SOURCE_HASHES,\n        "audit_source_sha256": sha(Path(__file__)),\n        "validation_id_order_sha256": VALIDATION_IDS_SHA,\n        "admitted_external_id_order_sha256": admitted_ids_sha,\n        "admitted_rows": len(admitted),\n        "admitted_prompt_blocks": int(admitted.prompt_id.nunique()),\n    }\n    admission_fingerprint = text_sha(json.dumps(fingerprint_inputs, sort_keys=True, separators=(",", ":")))\n    # The initial fixed pilot is exactly the previously audited 750 rows. A new\n    # exclusion changes the experimental input and must not silently launch it.\n    pass_frozen = len(admitted) == 750 and not excluded\n    receipt = {\n        "stage": "09", "status": "PASS_FROZEN_PILOT" if pass_frozen else "BLOCKED_FROZEN_SELECTION_CHANGED",\n        "dataset": "prvsiyan/prvsiyans-extended-name-that-llm-dataset", "dataset_version": 4,\n        "selection": "passages_clean.csv; split=new; provider_route=openai; five fixed GPT labels",\n        "official_train_sha256": TRAIN_SHA, "source_files_sha256": SOURCE_HASHES,\n        "audit_source_sha256": sha(Path(__file__)),\n        "admission_fingerprint": admission_fingerprint,\n        "admission_fingerprint_inputs": fingerprint_inputs,\n        "admitted_external_id_order_sha256": admitted_ids_sha,\n        "external_manifest_sha256": hashlib.sha256(manifest_bytes).hexdigest(),\n        "selected_rows": len(selected), "selected_prompt_blocks": int(selected.prompt_id.nunique()),\n        "admitted_rows": len(admitted), "admitted_prompt_blocks": int(admitted.prompt_id.nunique()),\n        "external_class_support": admitted.model.value_counts().sort_index().to_dict(),\n        "official_class_count": int(train.model.nunique()),\n        "external_class_count": int(admitted.model.nunique()),\n        "prompt_origin_declarations": prompt_rows.groupby(["prompt_source", "prompt_licence", "split"]).size().reset_index(name="prompt_blocks").to_dict("records"),\n        "blank_source_prompt_id_rows": int(selected.source_prompt_id.eq("").sum()),\n        "missing_prompt_bank_associations": len(missing),\n        "request_prompts_containing_mapped_prompt_body": int(body_match.sum()),\n        "selected_raw_call_envelopes_matched": len(evidence),\n        "raw_response_text_exact_rows": exact_raw,\n        "raw_response_comparison_policy": "whitespace normalization only; no text alteration for training",\n        "origin_excluded_blocks": origin_reasons, "overlap": overlap,\n        "exclusion_policy": "exclude every response in an external prompt block when any response or source association fails",\n        "split": split, "saved_stage07a_all_five_folds_compared_in_this_check": reference_checked,\n        "self_tests": self_tests(train),\n        "environment": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},\n        "limitations": [\n            "Official prompt wording is not supplied. Guessed/reconstructed official topics and response overlap cannot certify semantic prompt independence.",\n            "The actual mapped prompt bank declares own-written host-style prompts and CC BY 4.0; origin and call envelopes are publisher-supplied evidence, not independent provider attestation.",\n            "source_prompt_id is blank for all selected rows; association is corroborated through prompt_id, mapped prompt_body and the actual request envelope.",\n            "Admission is for this bounded private undistributed discriminative-classifier pilot; no generative fine-tuning, public trained weights or official deployment admission is inferred.",\n            "Only five external GPT labels are covered. All eighteen official labels remain in training and development evaluation.",\n            "The fixed official fold has been reused in earlier development work and is not a new untouched test.",\n        ],\n        "training_started": False, "provider_api_called": False, "paid_services_used": False,\n        "original_text_ids_prompt_ids_or_predictions_published": False,\n    }\n    if persist:\n        out.mkdir(parents=True, exist_ok=True)\n        (out / "external_manifest.csv").write_bytes(manifest_bytes)\n        (out / "split_receipt.json").write_text(json.dumps(split, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n        (out / "DATA_AUDIT.json").write_text(json.dumps(receipt, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n        lines = ["# Stage 09 data admission audit", "", "Status: **" + receipt["status"] + "**.", "",\n                 f"The immutable V4 selection contains {len(selected):,} direct OpenAI rows in {selected.prompt_id.nunique()} five-label prompt blocks. "\n                 f"{len(admitted):,} rows / {admitted.prompt_id.nunique()} blocks remain after whole-block source and response screens.", "",\n                 f"All {len(evidence)} selected published call envelopes match response, requested/served identity, actual request prompt, route and label. "\n                 f"All {int(body_match.sum())} actual prompts contain the mapped prompt-bank body. "\n                 "All selected prompt-bank records declare `own (written for this dataset, host-style protocol)` and `CC BY 4.0`.", "",\n                 f"Response screening found {overlap[\'flagged_rows\']} thresholded rows and excluded {len(excluded)} prompt blocks. "\n                 f"Maximum character cosine to any official response: {overlap[\'maximum_character_cosine\']:.6f}. "\n                 "Checks include exact/normalized/punctuation response equality, full-passage substring, at least three shared distinct word 8-grams and character cosine at least 0.85.", "",\n                 "Fold 1 is frozen at 2,159 training responses/120 prompts and 540 validation responses/30 prompts, with all 18 labels in each. "\n                 "Its ordered validation-ID hash matches the actual earlier Stage 07A fold. "\n                 + ("All five memberships were additionally compared to the saved Stage 07A OOF file in this check." if reference_checked else "Membership is enforced by the predeclared hash without requiring a row-level OOF attachment."), "",\n                 "**Admission is not semantic-independence certification.** Official prompt wording is unavailable. The own-written origin declaration and published call envelopes are evidence from the dataset publisher. "\n                 "The blank source_prompt_id field is documented; actual prompt-body/request association is checked. Unknown origins and explicit official reconstructions/train_regen derivatives are rejected as whole external blocks. "\n                 "The original fold remains reused development data.", "",\n                 "This audit does not train a classifier or use an inference API. It admits a bounded private discriminative-classifier experiment; it grants no public trained-weight or generative-model-use approval. "\n                 "Public artifacts contain aggregate results and external identifiers/hashes only.", "",\n                 f"Self-test fixtures: {receipt[\'self_tests\'][\'checks\']}/{receipt[\'self_tests\'][\'checks\']} passed. "\n                 "They exercise immutable-input tampering/missing files, unknown and reconstructed origins, train_regen, unknown licensing, invalid splits/empty prompts, and actual-call identity/prompt mismatches.", "",\n                 "Corpus, source, manifest and split hashes are in DATA_AUDIT.json and split_receipt.json.", ""]\n        (out / "DATA_AUDIT.md").write_text("\\n".join(lines), encoding="utf-8")\n    return train, admitted, receipt\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--train", type=Path)\n    parser.add_argument("--source", type=Path)\n    parser.add_argument("--out", type=Path)\n    parser.add_argument("--reference-oof", type=Path)\n    parser.add_argument("--self-test", action="store_true")\n    args = parser.parse_args()\n    if args.self_test:\n        print(json.dumps(self_tests(), indent=2))\n    else:\n        if not all([args.train, args.source, args.out]):\n            parser.error("--train, --source and --out are required")\n        _, _, result = audit(args.train, args.source, args.out, args.reference_oof)\n        print(json.dumps({key: result[key] for key in ["status", "selected_rows", "admitted_rows", "admitted_prompt_blocks",\n                         "selected_raw_call_envelopes_matched", "external_manifest_sha256", "self_tests"]}, indent=2))\n        require(result["status"] == "PASS_FROZEN_PILOT", "Frozen experiment blocked; retain audit and amend before training")\n', 'stage09_pretraining_pilot.py': '"""Stage 09: frozen, narrow external-data pretraining pilot; no test inference.\n\nThe two arms compare the staged procedure, including extra pretraining compute.\nThey do not isolate a compute-matched data effect or test the S03 ensemble.\nTorch/Transformers are imported only inside run(); --self-test is CPU-only.\n"""\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import Counter\nfrom datetime import datetime, timezone\nimport gc\nimport hashlib\nimport itertools\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport random\nimport time\nimport traceback\n\nimport numpy as np\nimport pandas as pd\n\nSEED = 20261024\nMODEL_ID = "microsoft/deberta-v3-base"\nREVISION = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"\nFOLD = 1\nOFFICIAL_SHA = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nMAX_LENGTH, BATCH_SIZE, ACCUMULATION = 384, 4, 4\nPRETRAIN_EPOCHS, OFFICIAL_EPOCHS = 2, 3\nPRETRAIN_ENCODER_LR, PRETRAIN_HEAD_LR = 1.5e-5, 1e-3\nOFFICIAL_ENCODER_LR, OFFICIAL_HEAD_LR = 1e-5, 5e-4\nGPU_BUDGET_SECONDS = 3 * 60 * 60\nPROTOCOL = {\n    "stage": "09", "seed": SEED, "fold": FOLD,\n    "model_id": MODEL_ID, "revision": REVISION,\n    "architecture": "AutoModel attention-mask mean of last hidden state + single Linear(hidden,18); no added dropout",\n    "max_length": MAX_LENGTH, "batch_size": BATCH_SIZE,\n    "gradient_accumulation": ACCUMULATION,\n    "pretrain_epochs": PRETRAIN_EPOCHS, "official_epochs": OFFICIAL_EPOCHS,\n    "pretrain_encoder_lr": PRETRAIN_ENCODER_LR, "pretrain_head_lr": PRETRAIN_HEAD_LR,\n    "official_encoder_lr": OFFICIAL_ENCODER_LR, "official_head_lr": OFFICIAL_HEAD_LR,\n    "official_train_rows": 2159, "official_validation_rows": 540,\n    "external_rows": 750, "external_classes": 5,\n    "pretrain_sampler": "all 750 admitted rows shuffled without replacement each epoch; exact 150 examples per each of five GPT classes",\n    "official_loss": "inverse-class-support weights normalized to mean one; sum weighted CE / accumulation-group actual rows",\n    "head_reset": "identical initial linear head in both official arms",\n    "optimizer": "fresh AdamW each phase; betas=(0.9,0.999), eps=1e-8, weight_decay=0.01 except biases/LayerNorm",\n    "scheduler": "linear decay, ceil(10% updates) warmup; new scheduler each phase",\n    "amp": "float32 parameters with fp16 autocast and GradScaler; any overflow fails phase",\n    "gradient_clip_norm": 1.0, "validation": "once, after fixed final official epoch; no early stopping",\n    "bootstrap": {"resamples": 10000, "unit": "official prompt_id", "seed": SEED},\n    "gpu_budget_seconds": GPU_BUDGET_SECONDS,\n    "required_allocated_gpus": 2,\n    "budget_unit": "allocated CUDA device count times wall seconds; cumulative across resumes",\n    "scope": "reused-development-fold pilot; narrow V4 direct-OpenAI five-GPT subset; not 83k upstream reproduction",\n    "comparison": "whole staged procedure including extra pretraining compute; not compute-matched data effect or S03 ensemble gain",\n}\n\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef file_sha(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef canonical_sha(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()\n\n\ndef save_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + "\\n")\n    temporary.replace(path)\n\n\ndef inverse_class_weights(y, classes):\n    support = np.bincount(np.asarray(y, dtype=int), minlength=classes)\n    if len(support) != classes or (support == 0).any():\n        raise ValueError("Official fit data must support every declared class")\n    weights = 1.0 / support\n    weights /= weights.mean()\n    return support, weights\n\n\ndef require_balanced_external(y):\n    counts = Counter(np.asarray(y, dtype=int).tolist())\n    if len(counts) != 5 or set(counts.values()) != {150}:\n        raise ValueError("Frozen external subset must contain exactly 150 rows in each of five classes")\n    return counts\n\n\ndef accumulation_groups(row_count, batch_size=BATCH_SIZE, accumulation=ACCUMULATION):\n    sizes = [min(batch_size, row_count - start) for start in range(0, row_count, batch_size)]\n    return [sizes[start:start + accumulation] for start in range(0, len(sizes), accumulation)]\n\n\ndef token_length_summary(lengths):\n    lengths = np.asarray(lengths, dtype=np.int64)\n    if lengths.ndim != 1 or len(lengths) == 0 or (lengths < 1).any():\n        raise ValueError("Token lengths must be a nonempty positive vector")\n    truncated = int((lengths > MAX_LENGTH).sum())\n    return {"rows": len(lengths), "truncated_rows": truncated,\n            "truncated_fraction": truncated / len(lengths),\n            "untruncated_token_length_quantiles": {\n                str(q): float(np.quantile(lengths, q)) for q in [0, .5, .9, .95, 1]},\n            "maximum_encoded_length": int(np.minimum(lengths, MAX_LENGTH).max())}\n\n\ndef validate_probabilities(probabilities, actual_ids, expected_ids, classes):\n    p = np.asarray(probabilities)\n    actual, expected = list(map(str, actual_ids)), list(map(str, expected_ids))\n    if len(set(actual)) != len(actual) or actual != expected:\n        raise ValueError("Validation row IDs must be unique and preserve frozen official order")\n    if (not np.issubdtype(p.dtype, np.number) or not np.isrealobj(p) or\n            p.shape != (len(expected), classes) or not np.isfinite(p).all()):\n        raise ValueError("Probability shape/finiteness check failed")\n    if (p < 0).any() or (p > 1).any() or not np.allclose(p.sum(1), 1, rtol=1e-5, atol=1e-6):\n        raise ValueError("Probabilities must lie in [0,1] and sum to one")\n    return p\n\n\ndef validate_prediction_payload(payload, expected_ids, labels):\n    if list(map(str, payload["labels"])) != list(labels):\n        raise ValueError("Saved probability columns have a different canonical label order")\n    p = validate_probabilities(payload["probabilities"], payload["ids"], expected_ids, len(labels))\n    logits = np.asarray(payload["logits"])\n    if (not np.issubdtype(logits.dtype, np.number) or not np.isrealobj(logits) or\n            logits.shape != p.shape or not np.isfinite(logits).all()):\n        raise ValueError("Saved logits have invalid shape/type or nonfinite values")\n    exponents = np.exp(logits.astype(np.float64) - logits.max(1, keepdims=True))\n    expected = exponents / exponents.sum(1, keepdims=True)\n    if not np.allclose(p, expected, rtol=1e-5, atol=1e-6):\n        raise ValueError("Saved probabilities differ from softmax of saved logits")\n    return p.copy()\n\n\ndef summarize_predictions(y, groups, control, candidate, labels):\n    """Aggregate only. Original IDs and row predictions never enter this result."""\n    from sklearn.metrics import balanced_accuracy_score, confusion_matrix\n    y, groups = np.asarray(y), np.asarray(groups)\n    if len(y) != len(groups) or set(np.unique(y)) != set(range(len(labels))):\n        raise ValueError("Validation support/alignment mismatch")\n    preds = {"control": np.asarray(control), "candidate": np.asarray(candidate)}\n    result, cms = {}, {}\n    for arm, pred in preds.items():\n        if pred.shape != y.shape or not np.isin(pred, np.arange(len(labels))).all():\n            raise ValueError("Prediction alignment/label mismatch")\n        cm = confusion_matrix(y, pred, labels=np.arange(len(labels)))\n        cms[arm] = cm\n        result[arm] = {"development_balanced_accuracy": float(balanced_accuracy_score(y, pred)),\n                       "confusion_matrix": cm.tolist(),\n                       "per_class": [{"label": lab, "support": int(cm[i].sum()),\n                                      "recall": float(cm[i, i] / cm[i].sum())}\n                                     for i, lab in enumerate(labels)]}\n    prompt_values = sorted(np.unique(groups).tolist())\n    gi = np.asarray([prompt_values.index(g) for g in groups])\n    total = np.zeros((len(prompt_values), len(labels)), dtype=np.int64)\n    np.add.at(total, (gi, y), 1)\n    draws = np.random.default_rng(SEED).multinomial(\n        len(prompt_values), np.full(len(prompt_values), 1 / len(prompt_values)), size=10000)\n    denominators = draws @ total\n    if (denominators == 0).any():\n        raise ValueError("Bootstrap draw omitted a class; do not silently change estimand")\n    values = {}\n    for arm, pred in preds.items():\n        correct = np.zeros_like(total)\n        np.add.at(correct, (gi, y), (pred == y).astype(np.int64))\n        values[arm] = ((draws @ correct) / denominators).mean(1)\n    delta = result["candidate"]["development_balanced_accuracy"] - result["control"]["development_balanced_accuracy"]\n    low, high = np.quantile(values["candidate"] - values["control"], [.025, .975])\n    good_c, good_t = preds["control"] == y, preds["candidate"] == y\n    result.update({"labels": labels, "validation_rows": len(y), "validation_prompts": len(prompt_values),\n                   "delta_percentage_points": 100 * delta,\n                   "paired_conditional_95_interval_percentage_points": [100 * float(low), 100 * float(high)],\n                   "bootstrap_resamples": 10000, "bootstrap_seed": SEED,\n                   "corrected": int((~good_c & good_t).sum()), "broken": int((good_c & ~good_t).sum()),\n                   "changed_predictions": int((preds["control"] != preds["candidate"]).sum()),\n                   "interpretation": "Reused fixed development fold, conditional on fitted models. No refitting/selection uncertainty. No automatic promotion, S03 ensemble gain, or official score claim.",\n                   "comparison_scope": PROTOCOL["comparison"]})\n    result["per_class_delta_percentage_points"] = [\n        {"label": lab, "delta": 100 * (result["candidate"]["per_class"][i]["recall"] - result["control"]["per_class"][i]["recall"])}\n        for i, lab in enumerate(labels)]\n    result["fixed_pilot_screen_pass"] = bool(delta > 0 and low > 0)\n    result["pilot_screen_scope"] = "Positive development BA difference and conditional paired 95% lower endpoint above zero earns broader grouped evaluation only. All class regressions retained; no automatic submission or ensemble claim."\n    return result\n\n\nclass GpuBudget:\n    def __init__(self, ledger_path, device_count, started=None):\n        self.path, self.devices = Path(ledger_path), int(device_count)\n        self.previous = json.loads(self.path.read_text())["gpu_seconds_consumed"] if self.path.exists() else 0.0\n        self.started = time.perf_counter() if started is None else started\n        if self.devices < 1:\n            raise ValueError("No allocated GPU")\n\n    def consumed(self):\n        return self.previous + self.devices * (time.perf_counter() - self.started)\n\n    def check(self, reserve_wall_seconds=120):\n        consumed = self.consumed()\n        save_json(self.path, {"gpu_seconds_consumed": consumed, "gpu_seconds_cap": GPU_BUDGET_SECONDS,\n                             "allocated_cuda_devices": self.devices, "updated_utc": utc_now(),\n                             "unit": PROTOCOL["budget_unit"]})\n        if consumed >= GPU_BUDGET_SECONDS - self.devices * reserve_wall_seconds:\n            raise TimeoutError("Frozen 3 GPU-hour budget guard reached; phase remains incomplete, schedule is not shortened")\n\n\ndef verify_resources(resource_dir, expected_manifest_sha):\n    resource_dir = Path(resource_dir)\n    manifest_path = resource_dir / "resources_manifest.json"\n    if file_sha(manifest_path) != expected_manifest_sha:\n        raise ValueError("Untouched vendor resource manifest SHA differs from prospective freeze")\n    receipt = json.loads(manifest_path.read_text())\n    if receipt.get("model_id") != MODEL_ID or receipt.get("revision") != REVISION:\n        raise ValueError("Wrong vendor model or revision")\n    model_dir = resource_dir / "deberta-v3-base"\n    expected = receipt["model_files_sha256"]\n    if not expected or "config.json" not in expected:\n        raise ValueError("Resource manifest omits model config")\n    if not any(name.endswith((".safetensors", ".bin")) for name in expected):\n        raise ValueError("Resource manifest omits weights")\n    for name, digest in expected.items():\n        relative = Path(name)\n        if relative.is_absolute() or ".." in relative.parts or not model_dir.joinpath(relative).is_file():\n            raise ValueError("Unsafe/missing model resource path")\n        if file_sha(model_dir / relative) != digest:\n            raise ValueError(f"Vendor resource changed: {name}")\n    # The original snapshot manifest intentionally excludes Hub cache metadata.\n    # Preserve that same inventory rule; cache receipts are not model inputs.\n    model_assets = {str(p.relative_to(model_dir)) for p in model_dir.rglob("*")\n                    if p.is_file() and ".cache" not in p.relative_to(model_dir).parts}\n    if model_assets != set(expected):\n        raise ValueError("Model asset inventory differs; no extra adapters/checkpoints permitted")\n    return model_dir, receipt\n\n\ndef run(train_path, source_dir, resource_dir, out_dir, freeze_path, resume=False, allocation_started=None):\n    # Offline inference APIs only; no Hub lookup, test file, paid API, or competition submission.\n    allocation_started = time.perf_counter() if allocation_started is None else allocation_started\n    os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", USE_TF="0", USE_FLAX="0",\n                      TOKENIZERS_PARALLELISM="false", CUBLAS_WORKSPACE_CONFIG=":4096:8")\n    import importlib.metadata\n    import torch\n    from torch import nn\n    import torch.nn.functional as F\n    from torch.utils.data import Dataset, DataLoader\n    from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup\n    from stage09_data_audit import audit, frozen_split\n\n    train_path, source_dir, resource_dir = map(Path, (train_path, source_dir, resource_dir))\n    out_dir, freeze_path = Path(out_dir), Path(freeze_path)\n    if out_dir.exists() and not resume:\n        raise FileExistsError("Preserve previous output; use explicit boundary-only --resume")\n    out_dir.mkdir(parents=True, exist_ok=True)\n    public, private = out_dir / "aggregate", out_dir / "private"\n    public.mkdir(exist_ok=True)\n    private.mkdir(exist_ok=True)\n    freeze = json.loads(freeze_path.read_text())\n    if freeze.get("status") != "FROZEN_BEFORE_GPU_TRAINING" or freeze.get("protocol") != PROTOCOL:\n        raise ValueError("Prospective source freeze missing or protocol differs")\n    if freeze.get("runner_sha256") != file_sha(__file__):\n        raise ValueError("Runner source differs from prospective freeze")\n    import stage09_data_audit\n    if freeze.get("audit_script_sha256") != file_sha(stage09_data_audit.__file__):\n        raise ValueError("Data-audit source differs from prospective freeze")\n    freeze_sha = file_sha(freeze_path)\n    status_path = public / "STATUS.json"\n    prior_state = json.loads(status_path.read_text()) if status_path.exists() else {}\n    if prior_state and prior_state.get("freeze_sha256") != freeze_sha:\n        raise ValueError("Resume freeze differs from original attempt")\n    assert torch.cuda.is_available(), "Included T4 CUDA GPU required; no CPU fallback"\n    allocated = torch.cuda.device_count()\n    assert allocated == PROTOCOL["required_allocated_gpus"], "Frozen hardware requires two allocated T4 GPUs"\n    assert all("T4" in torch.cuda.get_device_name(i) for i in range(allocated)), "Hardware amendment required"\n    budget = GpuBudget(private / "budget_ledger.json", allocated, started=allocation_started)\n    state = {"stage": "09", "status": "PREFLIGHT", "started_utc": prior_state.get("started_utc", utc_now()),\n             "freeze_sha256": freeze_sha, "completed_phases": [],\n             "actual_gpu_training": bool(prior_state.get("actual_gpu_training", False))}\n    save_json(status_path, state)\n    model = None\n\n    def reset_seed():\n        random.seed(SEED)\n        np.random.seed(SEED)\n        torch.manual_seed(SEED)\n        torch.cuda.manual_seed_all(SEED)\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n        torch.use_deterministic_algorithms(True, warn_only=True)\n\n    class MeanClassifier(nn.Module):\n        def __init__(self, model_dir):\n            super().__init__()\n            self.encoder = AutoModel.from_pretrained(model_dir, local_files_only=True).float()\n            self.head = nn.Linear(self.encoder.config.hidden_size, 18)\n\n        def forward(self, **inputs):\n            hidden = self.encoder(**inputs).last_hidden_state.float()\n            mask = inputs["attention_mask"].unsqueeze(-1).float()\n            pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1)\n            return self.head(pooled)\n\n    class TokenRows(Dataset):\n        def __init__(self, encoded, y, indices):\n            self.encoded, self.y, self.indices = encoded, y, np.asarray(indices)\n\n        def __len__(self):\n            return len(self.indices)\n\n        def __getitem__(self, pos):\n            i = int(self.indices[pos])\n            return {"inputs": {k: v[i] for k, v in self.encoded.items()}, "label": int(self.y[i]), "index": i}\n\n    def collate(rows):\n        values = tokenizer.pad([r["inputs"] for r in rows], padding=True, pad_to_multiple_of=8, return_tensors="pt")\n        return values, torch.tensor([r["label"] for r in rows]), [r["index"] for r in rows]\n\n    def checkpoint(phase, current_model, metadata):\n        # Only complete phases are resumable. Never load an incomplete or unverified checkpoint.\n        target = private / f"{phase}_COMPLETE"\n        if target.exists():\n            raise FileExistsError(f"Completed phase already exists: {phase}")\n        target.mkdir()\n        from safetensors.torch import save_file\n        # Independent clones also serialize correctly if a future upstream\n        # encoder exposes aliased state entries; no keys or tensors are omitted.\n        cpu_state = {k: v.detach().cpu().contiguous().clone() for k, v in current_model.state_dict().items()}\n        save_file(cpu_state, str(target / "model.safetensors"))\n        del cpu_state\n        save_json(target / "COMPLETE.json", {"phase": phase, "freeze_sha256": freeze_sha,\n                                            "model_sha256": file_sha(target / "model.safetensors"), **metadata})\n        return target\n\n    def load_complete(phase, current_model):\n        target = private / f"{phase}_COMPLETE"\n        if not target.exists():\n            return None\n        meta = json.loads((target / "COMPLETE.json").read_text())\n        if meta["phase"] != phase or meta["freeze_sha256"] != freeze_sha or file_sha(target / "model.safetensors") != meta["model_sha256"]:\n            raise ValueError("Completed phase checkpoint contract failed")\n        from safetensors.torch import load_file\n        current_model.load_state_dict(load_file(str(target / "model.safetensors"), device="cpu"), strict=True)\n        return meta\n\n    def parameter_groups(current_model, encoder_lr, head_lr):\n        groups = []\n        for prefix, module, lr in (("encoder", current_model.encoder, encoder_lr), ("head", current_model.head, head_lr)):\n            for no_decay in (False, True):\n                params = [p for name, p in module.named_parameters() if p.requires_grad and\n                          ((name.endswith("bias") or "LayerNorm" in name or "layer_norm" in name) == no_decay)]\n                if params:\n                    groups.append({"params": params, "lr": lr, "weight_decay": 0.0 if no_decay else 0.01})\n        assert sum(len(g["params"]) for g in groups) == sum(p.requires_grad for p in current_model.parameters())\n        return groups\n\n    def train_phase(phase, current_model, encoded, y, indices, epochs, encoder_lr, head_lr, pretraining=False):\n        existing = load_complete(phase, current_model)\n        if existing is not None:\n            print("RESUME_COMPLETE_PHASE", phase, flush=True)\n            return existing\n        reset_seed()\n        budget.check()\n        current_model.to("cuda:0")\n        data = TokenRows(encoded, y, indices)\n        generator = torch.Generator().manual_seed(SEED)\n        if pretraining:\n            require_balanced_external(y[indices])\n            loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True, generator=generator,\n                                num_workers=0, collate_fn=collate)\n            loss_weights = None\n        else:\n            support, weights = inverse_class_weights(y[indices], 18)\n            loss_weights = torch.tensor(weights, dtype=torch.float32, device="cuda:0")\n            loader = DataLoader(data, batch_size=BATCH_SIZE, shuffle=True, generator=generator,\n                                num_workers=0, collate_fn=collate)\n        total_steps = epochs * math.ceil(len(loader) / ACCUMULATION)\n        warmup = math.ceil(total_steps * .1)\n        optimizer = torch.optim.AdamW(parameter_groups(current_model, encoder_lr, head_lr), betas=(.9, .999), eps=1e-8)\n        scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)\n        scaler = torch.amp.GradScaler("cuda", init_scale=1024.0, growth_interval=2000)\n        rows_log, updates = [], 0\n        phase_started = time.perf_counter()\n        for epoch in range(1, epochs + 1):\n            current_model.train()\n            iterator, epoch_rows, loss_total = iter(loader), 0, 0.0\n            sampled_labels = Counter()\n            sampled_indices = []\n            while True:\n                batches = list(itertools.islice(iterator, ACCUMULATION))\n                if not batches:\n                    break\n                budget.check()\n                actual_rows = sum(len(batch[1]) for batch in batches)\n                optimizer.zero_grad(set_to_none=True)\n                for inputs, targets, batch_indices in batches:\n                    targets = targets.to("cuda:0")\n                    inputs = {k: v.to("cuda:0") for k, v in inputs.items()}\n                    with torch.autocast(device_type="cuda", dtype=torch.float16):\n                        logits = current_model(**inputs)\n                        loss_sum = F.cross_entropy(logits.float(), targets, weight=loss_weights, reduction="sum")\n                        loss = loss_sum / actual_rows\n                    if not torch.isfinite(loss) or not torch.isfinite(logits).all():\n                        raise FloatingPointError("Nonfinite training loss/logits; phase incomplete")\n                    if state["status"] != "RUNNING":\n                        first_started = state.get("first_training_backward_utc", utc_now())\n                        state.update(status="RUNNING", actual_gpu_training=True,\n                                     first_training_backward_utc=first_started)\n                        save_json(status_path, state)\n                    scaler.scale(loss).backward()\n                    loss_total += float(loss_sum.detach().cpu())\n                    epoch_rows += len(targets)\n                    sampled_labels.update(targets.detach().cpu().tolist())\n                    sampled_indices.extend(batch_indices)\n                scaler.unscale_(optimizer)\n                norm = torch.nn.utils.clip_grad_norm_(current_model.parameters(), 1.0)\n                prior_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                if scaler.get_scale() < prior_scale or not torch.isfinite(norm):\n                    raise FloatingPointError("AMP overflow/nonfinite gradient; no skipped step or automatic recipe change")\n                scheduler.step()\n                updates += 1\n                if updates % 25 == 0:\n                    print(f"phase={phase} epoch={epoch}/{epochs} update={updates}/{total_steps} gpu_seconds={budget.consumed():.1f}", flush=True)\n            assert epoch_rows == len(indices)\n            assert sorted(sampled_indices) == sorted(map(int, indices))\n            if pretraining:\n                assert sampled_labels == require_balanced_external(y[indices])\n            row = {"phase": phase, "epoch": epoch, "rows": epoch_rows,\n                   "sampled_class_counts": {labels[k]: v for k, v in sorted(sampled_labels.items())},\n                   "sample_order_sha256": canonical_sha(sampled_indices),\n                   "mean_weighted_loss": loss_total / epoch_rows, "cumulative_updates": updates,\n                   "validation_performed": False}\n            rows_log.append(row)\n            save_json(public / f"{phase}_training_log.json", rows_log)\n            print("EPOCH_COMPLETE", json.dumps(row), flush=True)\n        assert updates == total_steps\n        assert all(torch.isfinite(p).all() for p in current_model.parameters())\n        metadata = {"epochs": epochs, "optimizer_steps": updates, "warmup_steps": warmup,\n                    "seconds": time.perf_counter() - phase_started, "training_rows_per_epoch": len(indices),\n                    "encoder_lr": encoder_lr, "head_lr": head_lr, "amp_overflow_steps": 0}\n        checkpoint(phase, current_model, metadata)\n        budget.check()\n        del optimizer, scheduler, scaler\n        return metadata\n\n    def evaluate_once(arm, current_model):\n        target = private / f"{arm}_validation.npz"\n        receipt_path = private / f"{arm}_validation.json"\n        expected_ids = train.iloc[valid_idx].id.astype(str).tolist()\n        if target.exists():\n            meta = json.loads(receipt_path.read_text())\n            if meta["freeze_sha256"] != freeze_sha or meta["sha256"] != file_sha(target):\n                raise ValueError("Saved evaluation receipt differs")\n            with np.load(target, allow_pickle=False) as values:\n                return validate_prediction_payload(values, expected_ids, labels)\n        current_model.to("cuda:0").eval()\n        loader = DataLoader(TokenRows(official_encoded, y, valid_idx), batch_size=BATCH_SIZE,\n                            shuffle=False, num_workers=0, collate_fn=collate)\n        logits_values, indices = [], []\n        for inputs, _, batch_indices in loader:\n            budget.check()\n            with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):\n                logits = current_model(**{k: v.to("cuda:0") for k, v in inputs.items()})\n            if not torch.isfinite(logits).all():\n                raise FloatingPointError("Nonfinite final validation logits")\n            logits_values.append(logits.float().cpu().numpy())\n            indices.extend(batch_indices)\n        assert indices == valid_idx.tolist()\n        logits = np.concatenate(logits_values)\n        e = np.exp(logits - logits.max(1, keepdims=True))\n        probabilities = e / e.sum(1, keepdims=True)\n        validate_probabilities(probabilities, train.iloc[indices].id.tolist(), expected_ids, 18)\n        np.savez_compressed(target, ids=np.asarray(expected_ids), labels=np.asarray(labels),\n                            logits=logits, probabilities=probabilities)\n        save_json(receipt_path, {"freeze_sha256": freeze_sha, "sha256": file_sha(target),\n                                 "rows": len(expected_ids), "fixed_final_epoch": OFFICIAL_EPOCHS})\n        with np.load(target, allow_pickle=False) as values:\n            return validate_prediction_payload(values, expected_ids, labels)\n\n    try:\n        budget.check()\n        train, external, audit_receipt = audit(train_path, source_dir, public / "data_audit", persist=True)\n        if audit_receipt.get("status") != "PASS_FROZEN_PILOT" or audit_receipt.get("admission_fingerprint") != freeze["data_admission_fingerprint"]:\n            raise ValueError("Real CPU data-admission receipt differs from prospective freeze")\n        if file_sha(train_path) != OFFICIAL_SHA:\n            raise ValueError("Official train hash differs")\n        train_idx, valid_idx, split_receipt = frozen_split(train)\n        train_idx, valid_idx = np.asarray(train_idx), np.asarray(valid_idx)\n        assert len(train_idx) == 2159 and len(valid_idx) == 540\n        assert set(train.iloc[train_idx].prompt_id).isdisjoint(train.iloc[valid_idx].prompt_id)\n        labels = sorted(train.model.unique().tolist())\n        assert len(labels) == 18 and set(train.iloc[valid_idx].model) == set(train.iloc[train_idx].model) == set(labels)\n        y = train.model.map({label: i for i, label in enumerate(labels)}).to_numpy(dtype=int)\n        ey = external.model.map({label: i for i, label in enumerate(labels)}).to_numpy(dtype=int)\n        assert len(external) == 750 and len(np.unique(ey)) == 5 and not np.isnan(ey).any()\n        model_dir, resource_receipt = verify_resources(resource_dir, freeze["resource_manifest_sha256"])\n        environment = {"python": platform.python_version(),\n                       "packages": {name: importlib.metadata.version(name) for name in ["torch", "transformers", "sentencepiece", "numpy", "pandas", "scikit-learn", "safetensors"]},\n                       "cuda": torch.version.cuda, "allocated_gpu_count": allocated,\n                       "gpu_names": [torch.cuda.get_device_name(i) for i in range(allocated)],\n                       "training_device": "cuda:0", "all_allocated_devices_count_towards_budget": True,\n                       "determinism": "Seeded; deterministic algorithms warn-only; no bitwise GPU guarantee"}\n        assert environment["packages"]["transformers"] == "4.56.2", "Frozen transformer version required"\n        assert environment["packages"]["sentencepiece"] == "0.2.1", "Frozen sentencepiece required"\n        assert tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) >= (2, 6)\n        save_json(public / "environment.json", environment)\n        save_json(public / "protocol.json", PROTOCOL)\n        save_json(public / "resources_manifest.json", resource_receipt)\n        save_json(public / "label_order.json", labels)\n        tokenizer = AutoTokenizer.from_pretrained(model_dir, local_files_only=True, use_fast=False)\n        tokenizer.truncation_side = "right"\n        official_full_lengths = np.asarray([len(ids) for ids in tokenizer(\n            train.text.tolist(), truncation=False, padding=False)["input_ids"]])\n        external_full_lengths = np.asarray([len(ids) for ids in tokenizer(\n            external.text.tolist(), truncation=False, padding=False)["input_ids"]])\n        tokenization_receipt = {\n            "max_length": MAX_LENGTH, "use_fast": False, "truncation_side": "right",\n            "padding": "dynamic to multiple of 8", "special_tokens_included": True,\n            "official_training": token_length_summary(official_full_lengths[train_idx]),\n            "official_validation": token_length_summary(official_full_lengths[valid_idx]),\n            "external_pretraining": token_length_summary(external_full_lengths),\n        }\n        save_json(public / "tokenization.json", tokenization_receipt)\n        official_encoded = tokenizer(train.text.tolist(), truncation=True, max_length=MAX_LENGTH, padding=False)\n        external_encoded = tokenizer(external.text.tolist(), truncation=True, max_length=MAX_LENGTH, padding=False)\n        assert max(map(len, official_encoded["input_ids"])) <= MAX_LENGTH\n        assert max(map(len, external_encoded["input_ids"])) <= MAX_LENGTH\n        reset_seed()\n        model = MeanClassifier(model_dir)\n        initial_head = {k: v.detach().clone() for k, v in model.head.state_dict().items()}\n        initial_head_sha = hashlib.sha256(b"".join(initial_head[k].numpy().tobytes() for k in sorted(initial_head))).hexdigest()\n        save_json(public / "initial_head.json", {"sha256": initial_head_sha, "reset_before_both_official_arms": True})\n        # Exercise the real graph on two arbitrary fixtures before any fitting.\n        # No optimizer exists or is stepped, and fixture loss is not an accuracy result.\n        budget.check()\n        gpu_preflight = {"status": "RUNNING", "fixture_rows": 2, "no_optimizer_step": True,\n                         "actual_model_fitting": False, "initial_head_sha256": initial_head_sha}\n        save_json(public / "GPU_PREFLIGHT.json", gpu_preflight)\n        try:\n            model.to("cuda:0").eval()\n            model.zero_grad(set_to_none=True)\n            fixture = tokenizer(["Offline graph fixture one.", "A second brief graph fixture."],\n                                padding=True, pad_to_multiple_of=8, return_tensors="pt")\n            with torch.autocast(device_type="cuda", dtype=torch.float16):\n                fixture_logits = model(**{k: v.to("cuda:0") for k, v in fixture.items()})\n                fixture_loss = F.cross_entropy(fixture_logits.float(), torch.tensor([0, 1], device="cuda:0"))\n            assert fixture_logits.shape == (2, 18) and torch.isfinite(fixture_logits).all()\n            assert torch.isfinite(fixture_loss)\n            fixture_loss.backward()\n            used_gradients = [(name, p.grad) for name, p in model.named_parameters() if p.grad is not None]\n            assert used_gradients and all(torch.isfinite(g).all() for _, g in used_gradients)\n            assert any(name.startswith("encoder.") for name, _ in used_gradients)\n            assert any(name.startswith("head.") for name, _ in used_gradients)\n            head_after = model.head.state_dict()\n            head_after_sha = hashlib.sha256(b"".join(head_after[k].detach().cpu().numpy().tobytes()\n                                                     for k in sorted(head_after))).hexdigest()\n            assert head_after_sha == initial_head_sha, "No-step preflight changed classifier weights"\n            gpu_preflight.update(status="PASS_GRAPH_NO_OPTIMIZER_STEP", logit_shape=[2, 18],\n                                 logits_finite=True, loss_finite=True, used_gradient_tensors=len(used_gradients),\n                                 all_used_gradients_finite=True, head_after_sha256=head_after_sha,\n                                 head_unchanged=True, environment=environment)\n            model.zero_grad(set_to_none=True)\n            del fixture, fixture_logits, fixture_loss, used_gradients, head_after\n            torch.cuda.empty_cache()\n            save_json(public / "GPU_PREFLIGHT.json", gpu_preflight)\n            print("GPU_PREFLIGHT_PASS", json.dumps(gpu_preflight), flush=True)\n        except Exception as preflight_error:\n            gpu_preflight.update(status="FAIL_NO_FITTING", error_type=type(preflight_error).__name__,\n                                 error=str(preflight_error))\n            save_json(public / "GPU_PREFLIGHT.json", gpu_preflight)\n            raise\n        state["status"] = "PREFLIGHT_PASSED_READY_FOR_FITTING"\n        save_json(status_path, state)\n        phase_records = {}\n        phase_records["candidate_pretrain"] = train_phase("candidate_pretrain", model, external_encoded, ey,\n            np.arange(len(external)), PRETRAIN_EPOCHS, PRETRAIN_ENCODER_LR, PRETRAIN_HEAD_LR, pretraining=True)\n        state["completed_phases"].append("candidate_pretrain")\n        save_json(status_path, state)\n        del model\n        model = None\n        gc.collect()\n        torch.cuda.empty_cache()\n        reset_seed()\n        model = MeanClassifier(model_dir)\n        model.head.load_state_dict(initial_head, strict=True)\n        phase_records["control_official"] = train_phase("control_official", model, official_encoded, y,\n            train_idx, OFFICIAL_EPOCHS, OFFICIAL_ENCODER_LR, OFFICIAL_HEAD_LR)\n        control_probabilities = evaluate_once("control", model)\n        state["completed_phases"].append("control_official")\n        save_json(status_path, state)\n        del model\n        model = None\n        gc.collect()\n        torch.cuda.empty_cache()\n        reset_seed()\n        model = MeanClassifier(model_dir)\n        assert load_complete("candidate_pretrain", model) is not None\n        model.head.load_state_dict(initial_head, strict=True)\n        phase_records["candidate_official"] = train_phase("candidate_official", model, official_encoded, y,\n            train_idx, OFFICIAL_EPOCHS, OFFICIAL_ENCODER_LR, OFFICIAL_HEAD_LR)\n        candidate_probabilities = evaluate_once("candidate", model)\n        state["completed_phases"].append("candidate_official")\n        result = summarize_predictions(y[valid_idx], train.iloc[valid_idx].prompt_id.to_numpy(),\n            control_probabilities.argmax(1), candidate_probabilities.argmax(1), labels)\n        budget.check(reserve_wall_seconds=0)\n        result.update({"status": "COMPLETE_FIXED_FOLD_PILOT", "completed_utc": utc_now(), "protocol": PROTOCOL,\n                       "freeze_sha256": freeze_sha, "phase_records": phase_records,\n                       "gpu_seconds_consumed": budget.consumed(), "official_submission_made": False,\n                       "test_file_accessed": False, "actual_gpu_training": state["actual_gpu_training"],\n                       "tokenization": tokenization_receipt})\n        save_json(public / "RESULT.json", result)\n        pd.DataFrame({"id": train.iloc[valid_idx].id, "prompt_id": train.iloc[valid_idx].prompt_id,\n                      "model": train.iloc[valid_idx].model,\n                      "control": np.asarray(labels)[control_probabilities.argmax(1)],\n                      "candidate": np.asarray(labels)[candidate_probabilities.argmax(1)]}).to_csv(private / "validation_predictions.csv", index=False)\n        state.update(status="COMPLETE_FIXED_FOLD_PILOT", completed_utc=utc_now())\n        save_json(status_path, state)\n        print("STAGE09_COMPLETE", json.dumps(result, allow_nan=False), flush=True)\n        return result\n    except Exception as error:\n        state.update(status="INCOMPLETE_NO_RESULT", error_type=type(error).__name__, error=str(error), failed_utc=utc_now())\n        save_json(status_path, state)\n        (private / "failure_traceback.txt").write_text(traceback.format_exc())\n        budget.check(reserve_wall_seconds=0) if budget.consumed() < GPU_BUDGET_SECONDS else None\n        print("STAGE09_INCOMPLETE", json.dumps(state), flush=True)\n        raise\n    finally:\n        if model is not None:\n            del model\n        gc.collect()\n        torch.cuda.empty_cache()\n\n\ndef self_test():\n    checks = {}\n    groups = accumulation_groups(2159)\n    checks["official_exact_update_count_405"] = len(groups) * OFFICIAL_EPOCHS == 405\n    checks["official_last_accumulation_group_15_rows"] = groups[-1] == [4, 4, 4, 3]\n    checks["pretrain_exact_update_count_94"] = len(accumulation_groups(750)) * PRETRAIN_EPOCHS == 94\n    checks["pretrain_last_accumulation_group_14_rows"] = accumulation_groups(750)[-1] == [4, 4, 4, 2]\n    values = np.arange(1, 16, dtype=float) ** 2\n    reconstructed = sum(values[sum(groups[-1][:i]):sum(groups[-1][:i + 1])].sum() / 15 for i in range(4))\n    checks["partial_group_sum_loss_normalization_matches_example_mean"] = np.isclose(reconstructed, values.mean())\n    # Analytic scalar linear-model gradient, independent of the implementation\'s Torch path.\n    x, target, coef = np.arange(1., 16.), np.arange(15.) / 3, .7\n    gradient = 2 * (coef * x - target) * x\n    accumulated = sum(gradient[start:start + size].sum() / 15 for start, size in zip([0, 4, 8, 12], groups[-1]))\n    checks["partial_group_analytic_gradient_matches_full_batch"] = np.isclose(accumulated, gradient.mean())\n    y = np.repeat(np.arange(5), 150)\n    order = np.random.default_rng(SEED).permutation(len(y))\n    checks["external_full_shuffle_exact_balanced_class_exposure"] = require_balanced_external(y[order]) == Counter({i: 150 for i in range(5)})\n    checks["external_full_shuffle_no_duplicate_or_omitted_rows"] = sorted(order.tolist()) == list(range(750))\n    try:\n        require_balanced_external(y[:-1])\n    except ValueError:\n        checks["external_imbalanced_subset_rejected"] = True\n    else:\n        checks["external_imbalanced_subset_rejected"] = False\n    support, official_weights = inverse_class_weights(np.repeat(np.arange(18), np.arange(1, 19)), 18)\n    checks["official_weight_mass_balanced"] = np.allclose(support * official_weights, support[0] * official_weights[0])\n    checks["official_weights_mean_one"] = np.isclose(official_weights.mean(), 1)\n    lengths = token_length_summary([2, MAX_LENGTH, MAX_LENGTH + 1])\n    checks["truncation_incidence_uses_strict_boundary_including_specials"] = (\n        lengths["rows"] == 3 and lengths["truncated_rows"] == 1 and\n        lengths["truncated_fraction"] == 1 / 3 and lengths["maximum_encoded_length"] == MAX_LENGTH)\n    ids = ["a", "b", "c"]\n    valid = np.full((3, 18), 1 / 18)\n    checks["finite_aligned_probability_table_accepted"] = validate_probabilities(valid, ids, ids, 18).shape == (3, 18)\n    for name, p, actual in [("reordered_ids_rejected", valid, ids[::-1]),\n                            ("duplicate_ids_rejected", valid, ["a", "a", "c"]),\n                            ("nan_probability_rejected", np.full((3, 18), np.nan), ids),\n                            ("wrong_row_count_rejected", valid[:2], ids),\n                            ("unnormalized_probability_rejected", valid * 2, ids)]:\n        try:\n            validate_probabilities(p, actual, ids, 18)\n        except ValueError:\n            checks[name] = True\n        else:\n            checks[name] = False\n    payload = {"ids": ids, "labels": [f"label{i}" for i in range(18)],\n               "probabilities": valid, "logits": np.zeros((3, 18))}\n    checks["saved_payload_validated_including_logits"] = validate_prediction_payload(payload, ids, payload["labels"]).shape == (3, 18)\n    for name, field, value in [\n        ("saved_payload_column_order_rejected", "labels", payload["labels"][::-1]),\n        ("saved_payload_nonfinite_logits_rejected", "logits", np.full((3, 18), np.inf)),\n        ("saved_payload_probability_logit_mismatch_rejected", "logits", np.tile(np.arange(18), (3, 1))),\n    ]:\n        try:\n            validate_prediction_payload({**payload, field: value}, ids, payload["labels"])\n        except ValueError:\n            checks[name] = True\n        else:\n            checks[name] = False\n    # Every prompt has every class: exact 0-to-1 gain must remain +100 points in all bootstrap draws.\n    truth = np.tile(np.arange(18), 3)\n    summary = summarize_predictions(truth, np.repeat(["g0", "g1", "g2"], 18), (truth + 1) % 18, truth,\n                                    [f"label{i}" for i in range(18)])\n    checks["paired_bootstrap_known_100_point_gain"] = summary["delta_percentage_points"] == 100 and summary["paired_conditional_95_interval_percentage_points"] == [100, 100]\n    checks["paired_corrected_broken_known_contrast"] = summary["corrected"] == 54 and summary["broken"] == 0\n    checks = {name: bool(value) for name, value in checks.items()}\n    report = {"status": "PASS" if all(checks.values()) else "FAIL", "checks": checks,\n              "passed": sum(checks.values()), "total": len(checks),\n              "actual_gpu_training_executed": False,\n              "limits": ["CPU algebra/schema checks only; no Torch import, autograd, model load, AMP, CUDA, or GPU training executed.",\n                         "Real corpus admission is separately mandatory; synthetic checks do not establish data independence."]}\n    if not all(checks.values()):\n        raise AssertionError(json.dumps(report))\n    return report\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--self-test", action="store_true")\n    parser.add_argument("--train-path", type=Path)\n    parser.add_argument("--source-dir", type=Path)\n    parser.add_argument("--resource-dir", type=Path)\n    parser.add_argument("--out-dir", type=Path)\n    parser.add_argument("--freeze-path", type=Path)\n    parser.add_argument("--resume", action="store_true")\n    arguments = parser.parse_args()\n    if arguments.self_test:\n        print(json.dumps(self_test(), indent=2))\n    else:\n        required = [arguments.train_path, arguments.source_dir, arguments.resource_dir, arguments.out_dir, arguments.freeze_path]\n        if any(value is None for value in required):\n            parser.error("Training requires all five explicit paths; no automatic test/input discovery")\n        run(*required, resume=arguments.resume)\n'}
FREEZE = {'stage': '09', 'status': 'FROZEN_BEFORE_GPU_TRAINING', 'runner_sha256': '3f11efb611684130efba73ee7f5d245ad059e7e4ec6bfe5c158c9221989865a4', 'audit_script_sha256': '1a97c7bf5ac2d511c8c6ba4afbd9f4763e932f44daeda8808151e885cec72531', 'data_audit_receipt_sha256': '670e757a5b646900cfaacd5e89d02e88ffeea24826b351e4983fc29d94021a33', 'data_admission_fingerprint': 'efd4c7472f1b355940560095f9fbe31eebd649adcf80d4a82b89397defd779fa', 'resource_manifest_sha256': 'f8e60471afe49ec5f883bf141fc100d84081b2fe6a0eb28b1f9236486b7532bf', 'resource_source_script_version_id': 359317729, 'resource_source_url': 'https://www.kaggle.com/code/watchman777/name-that-llm-stage-04b-resources?scriptVersionId=359317729', 'protocol': {'stage': '09', 'seed': 20261024, 'fold': 1, 'model_id': 'microsoft/deberta-v3-base', 'revision': '8ccc9b6f36199bec6961081d44eb72fb3f7353f3', 'architecture': 'AutoModel attention-mask mean of last hidden state + single Linear(hidden,18); no added dropout', 'max_length': 384, 'batch_size': 4, 'gradient_accumulation': 4, 'pretrain_epochs': 2, 'official_epochs': 3, 'pretrain_encoder_lr': 1.5e-05, 'pretrain_head_lr': 0.001, 'official_encoder_lr': 1e-05, 'official_head_lr': 0.0005, 'official_train_rows': 2159, 'official_validation_rows': 540, 'external_rows': 750, 'external_classes': 5, 'pretrain_sampler': 'all 750 admitted rows shuffled without replacement each epoch; exact 150 examples per each of five GPT classes', 'official_loss': 'inverse-class-support weights normalized to mean one; sum weighted CE / accumulation-group actual rows', 'head_reset': 'identical initial linear head in both official arms', 'optimizer': 'fresh AdamW each phase; betas=(0.9,0.999), eps=1e-8, weight_decay=0.01 except biases/LayerNorm', 'scheduler': 'linear decay, ceil(10% updates) warmup; new scheduler each phase', 'amp': 'float32 parameters with fp16 autocast and GradScaler; any overflow fails phase', 'gradient_clip_norm': 1.0, 'validation': 'once, after fixed final official epoch; no early stopping', 'bootstrap': {'resamples': 10000, 'unit': 'official prompt_id', 'seed': 20261024}, 'gpu_budget_seconds': 10800, 'required_allocated_gpus': 2, 'budget_unit': 'allocated CUDA device count times wall seconds; cumulative across resumes', 'scope': 'reused-development-fold pilot; narrow V4 direct-OpenAI five-GPT subset; not 83k upstream reproduction', 'comparison': 'whole staged procedure including extra pretraining compute; not compute-matched data effect or S03 ensemble gain'}, 'training_started': False, 'source_freeze_note': 'Commit this source and receipt before any GPU fitting. Notebook and runner refuse source/input changes. Build time is not a training result.'}
# Embedded source is checksum-bound to the prospective receipt.
import sys
runtime_dir = Path("/kaggle/working/stage09_frozen_code")
runtime_dir.mkdir(exist_ok=True)
for filename, source in CODE_SOURCES.items():
    target = runtime_dir / filename
    target.write_text(source)
    expected = FREEZE["audit_script_sha256" if filename == "stage09_data_audit.py" else "runner_sha256"]
    if hashlib.sha256(target.read_bytes()).hexdigest() != expected:
        raise RuntimeError("Embedded frozen source mismatch: " + filename)
freeze_path = runtime_dir / "SOURCE_FREEZE.json"
freeze_path.write_text(json.dumps(FREEZE, indent=2, sort_keys=True, allow_nan=False) + "\n")
sys.path.insert(0, str(runtime_dir))
import stage09_pretraining_pilot as pilot
if pilot.PROTOCOL != FREEZE["protocol"]:
    raise RuntimeError("Embedded protocol mismatch")
print(json.dumps(pilot.self_test(), indent=2))


## Results

### 3. Execute the predeclared comparison

This is the only fitting cell. The runner creates completed phase-boundary checkpoints; it does not resume partial optimizer states or silently shorten a phase. A budget, AMP, numerical or contract failure leaves an explicit incomplete status.

Each arm is evaluated only after its fixed third official epoch. `aggregate/RESULT.json` retains development balanced accuracy, every class's recall and regression, confusion matrices, paired corrected/broken counts and the conditional paired prompt-bootstrap interval. Positive BA difference and a positive interval lower endpoint only earn broader grouped evaluation.

Original-row probabilities, logits, IDs and model checkpoints stay under `private/`. There is no test loading, submission or final-selection action.


In [ ]:
RUN_TRAINING = True
# The notebook must remain PRIVATE: checkpoints and original-row probabilities are private outputs.
if RUN_TRAINING:
    result = pilot.run(train_path, source_dir, resource_dir,
                       Path("/kaggle/working/stage09_pilot"), freeze_path,
                       resume=False, allocation_started=stage09_allocation_started)
    print("Fixed-fold development comparison completed; inspect aggregate/RESULT.json.")
else:
    print("PREPARED_ONLY: GPU training has not been enabled or executed.")


## Takeaways

No GPU score is populated in this prepared notebook. A completed fixed-fold result remains exploratory development evidence because these prompt groups have been reused. Public S03 accuracy, private leaderboard performance and an ensemble improvement are separate claims requiring their own evidence.

Source and data provenance: [Nishant Dahal part two V1](https://www.kaggle.com/code/nishantdahal/two-stage-lora-part-two?scriptVersionId=361311282) motivated the staged mean-pooling encoder architecture/rates; [prvsiyan's Extended Dataset V4](https://www.kaggle.com/datasets/prvsiyan/prvsiyans-extended-name-that-llm-dataset/versions/4), source-declared CC BY 4.0, supplies the narrow externally admitted corpus; [our pinned vendor resource preparation](https://www.kaggle.com/code/watchman777/name-that-llm-stage-04b-resources?scriptVersionId=359317729) supplies the untouched MIT-licensed DeBERTa-v3-base revision. No upstream adapted checkpoint initializes these development arms.
